# 🌦️🔋 AfyaSolar AI Engine — Training Notebook

Train every model the AI engine serves, end to end, in one place:

| Track | Models | Compute |
|-------|--------|---------|
| **Climate** | Chronos-Bolt fine-tuned on NASA POWER (daily + monthly) | **GPU** |
| **Predictive maintenance** | XGBoost RUL · Isolation-Forest anomaly | CPU |

Everything is open and free: NASA POWER (open data), Chronos-Bolt (Apache-2.0), AutoGluon (Apache-2.0).

**How to use**
1. Runtime → Change runtime type → **T4 GPU**.
2. Run the cells top to bottom. Edit **Section 2 (Configuration)** to trade speed vs. accuracy.
3. Download the trained models at the end (Section 6).

> The maintenance track (Section 5) is CPU-only — you can run it without a GPU.

---
## 0 · Runtime check

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(out or "⚠️  No GPU detected — set Runtime → Change runtime type → T4 GPU.\n"
      "    (Section 5 / maintenance still works on CPU.)")

---
## 1 · Get the code

In [ ]:
# The AfyaSolar Intelligence monorepo. For a private repo use a token:
#   https://<TOKEN>@github.com/<owner>/afyasolar.git
REPO_URL = "https://github.com/ubuntuafyalink/afyasolar.git"  # <-- change if needed

import os
if not os.path.isdir("afyasolar"):
    !git clone $REPO_URL afyasolar
%cd afyasolar/ai-service
print("\nworking dir:", os.getcwd())

---
## 2 · Configuration
Edit these, then run. Smaller model / fewer steps / shorter history = faster.

In [ ]:
# --- Climate (Chronos) ---
START_DATE      = "19900101"   # NASA history start (YYYYMMDD). Use 20050101 for a quick run.
CHRONOS_MODEL   = "bolt_base"  # bolt_small (fast) | bolt_base (accurate)
FINE_TUNE_STEPS = 2000         # 500 for a quick smoke test

# --- Predictive maintenance (synthetic telemetry) ---
MAINT_FACILITIES = 40
MAINT_DAYS       = 2200

print(f"climate: {CHRONOS_MODEL}, {FINE_TUNE_STEPS} steps, history from {START_DATE}")
print(f"maintenance: {MAINT_FACILITIES} facilities x {MAINT_DAYS} days")

In [ ]:
# Install the training stack (AutoGluon + Chronos + XGBoost + scikit-learn).
# Takes a few minutes the first time.
!pip install -q -r pipeline/train/requirements.txt

In [ ]:
# Apply the config choices to pipeline/train/config.yaml
import yaml
cfg = yaml.safe_load(open("pipeline/train/config.yaml"))
cfg["model"]["chronos_model"] = CHRONOS_MODEL
cfg["model"]["fine_tune_steps"] = int(FINE_TUNE_STEPS)
yaml.safe_dump(cfg, open("pipeline/train/config.yaml", "w"), sort_keys=False)
print("updated config:", cfg["model"])

---
## 3 · Climate model — Chronos  🌦️  (GPU)
Chronos forecasts the **raw** NASA variables; the app derives hazards and solar yield from them.

### 3.1 · Fetch NASA POWER (34 locations, 7 variables)

In [ ]:
!python pipeline/data/fetch_nasa.py --start $START_DATE

### 3.2 · Build the Chronos-ready datasets (daily + monthly)

In [ ]:
!python pipeline/datasets/build_dataset.py
import json
print("config:", json.load(open("pipeline/datasets/processed/dataset_summary.json"))["config"])

### 3.3 · Fine-tune Chronos-Bolt
Trains **SeasonalNaive** (baseline) vs **Chronos ZeroShot** vs **Chronos FineTuned** per horizon. ~20–40 min for the defaults.

In [ ]:
!python pipeline/train/finetune_chronos.py --horizon both

### 3.4 · Backtest & report

In [ ]:
!python pipeline/eval/backtest.py --horizon both
from pathlib import Path
from IPython.display import Markdown, display
for h in ["daily", "monthly"]:
    rep = Path(f"pipeline/train/outputs/{h}/backtest_report.md")
    if rep.exists():
        display(Markdown(rep.read_text()))

---
## 4 · Predictive maintenance  🔋  (CPU)
Trained on **synthetic** telemetry (physics-based battery ageing + injected faults), since no live device data exists yet. Retrain on real telemetry when it arrives — the interfaces don't change.

### 4.1 · Generate labeled telemetry

In [ ]:
!python pipeline/synthetic/generate_telemetry.py --facilities $MAINT_FACILITIES --days $MAINT_DAYS

### 4.2 · Train the RUL model (XGBoost + SHAP)

In [ ]:
!python pipeline/train/train_rul.py

### 4.3 · Train the anomaly detector (Isolation Forest)

In [ ]:
!python pipeline/train/train_anomaly.py

---
## 5 · Results summary

In [ ]:
import json
from pathlib import Path

def show(path, label):
    p = Path(path)
    print(f"{label:22s}", json.load(open(p)) if p.exists() else "(not trained)")

print("=== Climate (Chronos) — leaderboards saved per horizon ===")
for h in ["daily", "monthly"]:
    lb = Path(f"pipeline/train/outputs/{h}/leaderboard.csv")
    print(f"{h:22s}", "leaderboard.csv ✓" if lb.exists() else "(not trained)")
print("\n=== Predictive maintenance ===")
show("pipeline/train/outputs/rul/metrics.json", "RUL")
show("pipeline/train/outputs/anomaly/metrics.json", "Anomaly")

---
## 6 · Download the trained models
Zip everything under `pipeline/train/outputs/` and download it. Point the API's `AI_ENGINE_MODEL_DIR` at these, or push them to a HuggingFace repo.

In [ ]:
!cd pipeline/train && zip -qr outputs.zip outputs && echo 'created pipeline/train/outputs.zip'
try:
    from google.colab import files
    files.download('pipeline/train/outputs.zip')
except Exception as e:
    print('Not on Colab or download unavailable:', e)

#### (optional) Push the fine-tuned models to HuggingFace
Needs an `HF_TOKEN` with **Write** scope (https://huggingface.co/settings/tokens).

In [ ]:
# from huggingface_hub import HfApi, login
# login(token="hf_...")            # or set the HF_TOKEN env var
# HfApi().upload_folder(
#     folder_path="pipeline/train/outputs",
#     repo_id="<your-username>/afyasolar-models",
#     repo_type="model",
# )